In [ ]:
from PIL import Image
import numpy as np
import os
from constants import ImageMetadata, PIPE_PATH
import json
from tkinter import filedialog
from tkinter import Tk

In [ ]:
def load_metadata_from_dialog():
    # Hide the main window
    root = Tk()
    root.withdraw()

    # Open file dialog
    image_path = filedialog.askopenfilename(
        title="Select a file",
        filetypes=[("Gray image files", "*.ImageMetadata"), ("Color image files", "*.ppm")]
    )

    if not image_path:
        print("No file selected")
        return None

    with Image.open(image_path) as img:
        arr = np.array(img)
        data = arr.tobytes()
        return ImageMetadata(img.height, img.width, int(arr.max()), data)

In [ ]:
def ask_metadata_and_start_pipe(metadata):
    if metadata is None:
        return

    if not os.path.exists(PIPE_PATH):
        os.mkfifo(PIPE_PATH)
        print(f"Created named pipe at {PIPE_PATH}")

    print("Writer process started. Opening pipe...")

    with open(PIPE_PATH, "wb") as pipe:
        message = json.dumps(metadata.to_dict()).encode("utf-8")
        pipe.write(message + b"\n")
        pipe.write(metadata.data)
        pipe.flush()

    print("Writer process finished")

In [ ]:
def main():
    metadata = load_metadata_from_dialog()
    if metadata:
        ask_metadata_and_start_pipe()